# Start a Local Cluster

In [101]:
from functools import reduce
from pyspark.sql.functions import (col, trim, lower, regexp_replace, sum, udf, to_timestamp,split, datediff, substring, length,
    current_timestamp, when, datediff, try_to_timestamp, to_date)
from pythainlp import word_tokenize
from pyspark.sql.types import ArrayType, StringType
from pythainlp.corpus import thai_stopwords




In [102]:
spark_url = 'local'
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

spark = SparkSession.builder \
    .appName("TraffyFondueDataCleaning") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [103]:
sc = spark.sparkContext
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT"))


RAW_DIR = PROJECT_ROOT / "data" / "raw" / "traffy-fondue"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "traffy-fondue"

file_path = (RAW_DIR/'bangkok.csv')

## schema

In [ ]:

traffy_schema = StructType([
    # ตัวระบุเฉพาะ
    StructField("ticket_id", StringType(), True),
    
    # ข้อมูลปัญหาและการจัดการ
    StructField("type", StringType(), True),         
    StructField("organization", StringType(), True), 
    StructField("comment", StringType(), True),      
    StructField("photo", StringType(), True),
    StructField("photo_after", StringType(), True),
    
    # ข้อมูลพิกัดและตำแหน่ง
    StructField("coords", StringType(), True),      
    StructField("address", StringType(), True),
    StructField("subdistrict", StringType(), True),
    StructField("district", StringType(), True),
    StructField("province", StringType(), True),
    
    # ข้อมูลเวลาและสถานะ
    StructField("timestamp", StringType(), True),    
    StructField("state", StringType(), True),        
    
    # ข้อมูลการตอบรับและกิจกรรม
    StructField("star", FloatType(), True),         
    StructField("count_reopen", IntegerType(), True), 
    StructField("last_activity", StringType(), True)  
])

In [ ]:
df_traffy = spark.read.csv(
    str(file_path),
    header=True,
    schema=traffy_schema,
    multiLine=True, 
    escape='"' 
)

In [ ]:
# ลิสต์หมวดหมู่หลักที่ส่งผลต่อมูลค่าอสังหาฯ และความน่าอยู่
livability_types = [
    "ถนน",
    "ทางเท้า",
    "ความปลอดภัย",
    "แสงสว่าง",
    "ความสะอาด",
    "กีดขวาง",
    "ท่อระบายน้ำ",
    "น้ำท่วม",
    "ต้นไม้",
    "PM2",
    "จราจร",
    "สะพาน"
]

df_filtered_type = df_traffy.filter(
    reduce(lambda a, b: a | b, [col("type").contains(t) for t in livability_types])
)

active_states = [
    "กำลังดำเนินการ",
    "รอรับเรื่อง" 
]

from pyspark.sql.functions import trim, col

df_filtered_type = df_filtered_type.withColumn(
    "state", trim(col("state"))
)

df_filtered_type = df_filtered_type.filter(
    col("state").isin(active_states)
)



In [ ]:
df_final_spatial = df_filtered_type.withColumn(
    "lon_raw", 
    trim(split(col("coords"), ",").getItem(0)).cast("float")
).withColumn(
    "lat_raw", 
    trim(split(col("coords"), ",").getItem(1)).cast("float")
).withColumn(
    "lon", col("lon_raw")
).withColumn(
    "lat", col("lat_raw")
).filter(
    (col("lat") >= 5) & (col("lat") <= 21) & 
    (col("lon") >= 97) & (col("lon") <= 105)
)
BANGKOK_PROVINCE_NAMES = ["กรุงเทพมหานคร", "กรุงเทพ","จังหวัดกรุงเทพมหานคร"]

df_bangkok_only = df_final_spatial.filter(
    col("province").isin(BANGKOK_PROVINCE_NAMES)
)



In [ ]:
df_trim_string = df_bangkok_only.withColumn(
    "timestamp_str", 
    substring(col("timestamp"), 1, 19)
).withColumn(
    "last_activity_str", 
    substring(col("last_activity"), 1, 19)
)

TIMESTAMP_FORMAT_SIMPLE = "yyyy-MM-dd HH:mm:ss"

df_time_prep = df_trim_string.withColumn(
    "timestamp_dt", 
    to_timestamp(col("timestamp_str"), TIMESTAMP_FORMAT_SIMPLE)
).withColumn(
    "last_activity_dt", 
    to_timestamp(col("last_activity_str"), TIMESTAMP_FORMAT_SIMPLE)
)

df_time_prep = df_time_prep.filter(
    col("timestamp_dt").isNotNull() 
)

df_time_prep = df_time_prep.withColumn("timestamp_date", to_date(col("timestamp_dt")))
df_time_prep = df_time_prep.withColumn("last_activity_date", to_date(col("last_activity_dt")))

df_final_ready = df_time_prep.withColumn(
    "DaysToFix",
    when(
        col("state") == "เสร็จสิ้น",
        datediff(col("last_activity_date"), col("timestamp_date"))
    ).otherwise(
        datediff(to_date(current_timestamp()), col("timestamp_date"))
    )
)

In [109]:

df_clean = (
    df_final_ready
    .withColumn("comment_clean", trim(col("comment")))
    .withColumn("comment_clean", lower(col("comment_clean")))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), "[\n\r\t]", " "))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), "[^ก-๙a-z0-9/. ]", ""))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), " +", " "))
)

MIN_COMMENT_LENGTH = 10
df_clean = df_clean.filter(
    (length(col("comment_clean")) >= MIN_COMMENT_LENGTH)
)

In [ ]:
from pyspark.sql.functions import col, desc

df_comment_counts = df_clean.groupBy("comment_clean").count()

df_duplicate_summary = df_comment_counts.orderBy(
    col("count").desc()
)


In [ ]:
from pyspark.sql.functions import col

comment_to_filter = "ทางเท้าชำรุด"
df_damaged_pavement = df_clean.filter(col("comment_clean") == comment_to_filter)

df_analysis_subset = df_damaged_pavement.select(
    "ticket_id",
    "timestamp_dt",
    "last_activity_dt",
    "lat",
    "lon",
    "district",
    "DaysToFix",
)

In [ ]:
from pyspark.sql.functions import col, floor, row_number, min
from pyspark.sql.window import Window

df_grouped = df_clean.withColumn("micro_lat", floor(col("lat") * 10000)) \
                        .withColumn("micro_lon", floor(col("lon") * 10000)) \
                        .withColumn("comment_group", col("comment_clean"))

window_spec = Window.partitionBy("micro_lat", "micro_lon", "comment_group").orderBy(col("timestamp_dt").asc())

df_ranked = df_grouped.withColumn(
    "rank", 
    row_number().over(window_spec)
)

df_deduplicated_final = df_ranked.filter(col("rank") == 1).drop("micro_lat", "micro_lon", "comment_group", "rank")

In [ ]:
from pyspark.sql.functions import col, desc

df_comment_counts = df_deduplicated_final.groupBy("comment_clean").count()

df_duplicate_summary = df_comment_counts.orderBy(
    col("count").desc()
)

In [ ]:
COLUMNS_TO_KEEP = [
    "ticket_id",
    "type",
    "address",
    "district",
    
    "lat",
    "lon",
    "DaysToFix",
    
    "comment_clean",
    
    "timestamp_dt",
    "last_activity_dt"
]

df_ready_for_export = df_deduplicated_final.select(*COLUMNS_TO_KEEP)
df_ready_for_export = df_ready_for_export.na.drop(subset=["comment_clean", "district"])
df_ready_for_export = df_ready_for_export.filter(
    col("district").isNotNull() 
)

In [ ]:
from pyspark.sql.functions import lit

df_multi_hot = df_ready_for_export

for t in livability_types:
    new_col_name = f"type_{t}"
    
    df_multi_hot = df_multi_hot.withColumn(
        new_col_name,
        when(col("type").contains(t), lit(1)).otherwise(lit(0))
    )

selected_cols = ["ticket_id", "type"] + [f"type_{t}" for t in livability_types]
selected_cols.pop(-3)
selected_cols.append("type_PM25")
df_multi_hot = df_multi_hot.withColumnRenamed("type_PM2", "type_PM25")

In [ ]:
from pyspark.sql.functions import col, year

df_final_year = df_multi_hot.withColumn(
    "year_reported", 
    year(col("timestamp_dt"))
)

df_final_year = df_final_year.withColumn(
    "year_last_activity", 
    year(col("last_activity_dt"))
)

In [ ]:
df_final_ml = df_final_year.drop("type") 
df_final_ml = df_final_ml.withColumnRenamed("DaysToFix", "DaysActive_Pending")


In [ ]:


from pyspark.sql.functions import col, regexp_replace

string_cols_to_clean = ["comment_clean", "address", "district"] 

df_cleaned_for_export = df_final_ml 

for col_name in string_cols_to_clean:
    df_cleaned_for_export = df_cleaned_for_export.withColumn(
        col_name,
        regexp_replace(col(col_name), "[\r\n]", " ")
    )

print("--- ล้างอักขระขึ้นบรรทัดใหม่เสร็จสิ้น ---")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "traffy-fondue"
OUTPUT_PATH = PROCESSED_DIR/"traffy_fondue_bangkok_processed.csv"


pdf = df_cleaned_for_export.toPandas()

pdf.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("✅ Saved CSV to:", OUTPUT_PATH)


--- ล้างอักขระขึ้นบรรทัดใหม่เสร็จสิ้น ---
✅ Saved CSV to: C:\Users\sasit\CU\2-1\dsde\project\dsdengdeng-project-dsde\data\processed\traffy-fondue\traffy_fondue_bangkok_processed.csv


In [ ]:
from pyspark.sql.functions import rand

df_sample = df_final_ml.orderBy(rand(seed=42)).limit(10000)
df_map_sample_pandas = df_sample.toPandas()
display(df_map_sample_pandas.head(10))

,ticket_id,address,district,lat,lon,DaysActive_Pending,comment_clean,timestamp_dt,last_activity_dt,type_ถนน,...,type_ความสะอาด,type_กีดขวาง,type_ท่อระบายน้ำ,type_น้ำท่วม,type_ต้นไม้,type_PM25,type_จราจร,type_สะพาน,year_reported,year_last_activity
0,2025-D8ATWX,VJQC+C9R แขวงคลองถนน เขตสายไหม กรุงเทพมหานคร 1...,สายไหม,13.88856,100.620911,5,ท่อระบายน้ำเออล้น ความสูงระดับไม่สูงแต่เออขึ้น...,2025-12-02 13:16:09,2025-12-02 13:27:45,0,...,0,0,0,1,0,0,0,0,2025,2025
1,2025-RUPX9E,RHCM+VXW ซอย รัชดาภิเษก 36 แยก 9-15 แขวงจันทรเ...,จตุจักร,13.82231,100.584976,1,รถจอดถนนสาธารณะกีดขวางการจราจรร้องเรียนไปหลายร...,2025-12-06 14:04:52,2025-12-06 14:25:34,0,...,0,0,0,0,0,0,1,0,2025,2025
2,2025-69EKTM,10/648 ถ. กาญจนาภิเษก แขวงหลักสอง บางแค กรุงเท...,บางแค,13.69554,100.404610,3,รถวิ่งเร็วหลังจากทำถนนใหม่เสร็จ เป็นสี่แยกมีคน...,2025-12-04 08:13:10,2025-12-04 14:17:15,1,...,0,0,0,0,0,0,0,0,2025,2025
3,2025-7VPC9N,48 ถ. จรัญสนิทวงศ์ แขวงบางพลัด เขตบางกอกน้อย ก...,บางพลัด,13.78808,100.500427,4,จอดรถจักรยานยนต์บนทางเท้าจุดสังเกตอยู่บริเวณหน...,2025-12-03 10:02:42,2025-12-03 10:07:57,0,...,0,0,0,0,0,0,0,0,2025,2025
4,2025-H9YCTA,2010/3 ซ. พหลโยธิน แขวงเสนานิคม เขตจตุจักร กรุ...,จตุจักร,13.83594,100.573463,5,ขอทราบเหตุผลการไม่ห้ามร้านนี้ใช้พื้นที่ทางเท้า...,2025-12-02 19:58:39,2025-12-03 21:10:36,0,...,0,0,0,0,0,0,0,0,2025,2025
5,2025-FCZAFP,แผง3-4 ถ. สามเสน แขวงวัดสามพระยา เขตพระนคร กรุ...,พระนคร,13.76978,100.503479,1,นี้หรือ... แก้ไขแล้ว...ครับ ท่าน ก็ยังเห็น ตั้...,2025-12-06 09:27:07,2025-12-07 00:08:49,0,...,0,0,0,0,0,0,0,0,2025,2025
6,2025-X8RD6F,43/1 พึ่งมี​ 29​ แยก​ 3 แขวงบางจาก เขตพระโขนง ...,พระโขนง,13.70289,100.619453,6,ปัญหา กลิ่นควันที่เกิดจากการเผาไหม้ถ่าน ที่คาด...,2025-12-01 19:24:46,2025-12-03 09:55:04,0,...,0,0,0,0,0,1,0,0,2025,2025
7,2025-N6UAEH,เลขที่ 785 1 ถ. ประชาอุทิศ แขวงสามเสนนอก เขตห้...,ห้วยขวาง,13.76922,100.591789,2,แจ้งถนนชำรุด แตกพังสูงชันและแตกพังเป็นหลุม ตรง...,2025-12-05 15:40:22,2025-12-05 18:25:33,1,...,0,0,0,0,0,0,0,0,2025,2025
8,2025-48CYVM,287 ซอย อมรวิวัฒน์ แขวง คันนายาว เขตคันนายาว ก...,คันนายาว,13.83620,100.661407,3,มีรถแท็กซี่ จอดบริเวณเส้นขาวแดง ทางเข้าคอนโด จ...,2025-12-04 09:31:59,2025-12-04 10:59:47,0,...,0,0,0,0,0,0,1,0,2025,2025
9,2025-33G4E7,"Bang Bamru Station, แขวงบางพลัด บางพลัด กรุงเท...",บางพลัด,13.79156,100.477898,4,เส้นถนนจางมากทำให้รถไม่รู้เลนตนเอง ไม่รู้ว่าต้...,2025-12-03 16:32:49,2025-12-05 08:17:20,1,...,0,0,0,0,0,0,0,0,2025,2025


In [ ]:
from pymongo import MongoClient
import pandas as pd



MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db = client["my_project"]
collection = db["traffic_clean"]


PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "traffy-fondue"
df = pd.read_csv(PROCESSED_DIR/'traffy_fondue_bangkok_processed.csv')



csv_ids = df["ticket_id"].astype(str).tolist()

print(f"📌 CSV มีทั้งหมด {len(csv_ids)} แถว")

existing = collection.find(
    {"ticket_id": {"$in": csv_ids}},
    {"ticket_id": 1, "_id": 0}
)

existing_ids = {item["ticket_id"] for item in existing}

print(f"🔍 พบข้อมูลซ้ำใน MongoDB จำนวน {len(existing_ids)} รายการ")

df_unique = df[~df["ticket_id"].astype(str).isin(existing_ids)]

print(f"✨ จะทำการ insert เฉพาะข้อมูลใหม่จำนวน {len(df_unique)} รายการ")

data_dict = df_unique.to_dict("records")

if len(data_dict) > 0:
    try:
        collection.insert_many(data_dict)
        print("✅ Insert สำเร็จ!")
    except Exception as e:
        print("❌ เกิดข้อผิดพลาดขณะ insert:", e)
else:
    print("⚠️ ไม่มีข้อมูลใหม่ให้เพิ่ม (ทั้งหมดซ้ำ)")

📌 CSV มีทั้งหมด 1898 แถว
🔍 พบข้อมูลซ้ำใน MongoDB จำนวน 1898 รายการ
✨ จะทำการ insert เฉพาะข้อมูลใหม่จำนวน 0 รายการ
⚠️ ไม่มีข้อมูลใหม่ให้เพิ่ม (ทั้งหมดซ้ำ)
